# v09 · Callimachus v0.9: Unicode public charset, SHA label as primary key, SQLite store
`PtolemyDesktop/Callimachus/v09/` (git 2026-05-02).

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
import sys
sys.path.insert(0, os.path.join(PROJ, "PtolemyDesktop"))
from Callimachus.v09.core.hyperwebster import HyperWebster as HW9
from Callimachus.v09.core import charset as cs
from Callimachus.v09.core.lsh_datatype import LSH_Datatype
from Callimachus.v09.database.hyperdatabase import HyperDatabase
from Callimachus.v09.gallery import hypergallery as hg9

## Core: `IndexRecord(label, payload, length, timestamp)`

In [3]:
hw = HW9()
rec = hw.index("Hello, World!")
print(rec.label, "|", repr(rec.payload), "|", rec.length)
assert hw.regenerate(rec.payload, rec.length) == "Hello, World!"
print("N =", hw.N, "| first three symbols:", [repr(c) for c in cs.PUBLIC_CHARSET[:3]], "| last:", repr(cs.PUBLIC_CHARSET[-1]))
for size in (140, 1000, 4000):
    r = hw.index(CORPUS[:size]); assert hw.regenerate(r.payload, r.length) == CORPUS[:size]
print("round trips exact at 140/1000/4000 chars")

41fdb0d8d378479e80492421151ae9ae06abfaa280e29df6d50b68a2c66fc713 | 'Ifmmp-!Xpsme!' | 13
N = 97 | first three symbols: ["'\\t'", "'\\n'", "' '"] | last: '~'
round trips exact at 140/1000/4000 chars


## Database: label is the primary key; `first_encountered` lives on the datatype

In [4]:
db = HyperDatabase(":memory:")
lsh = LSH_Datatype.from_acquisition(rec.label, "Hello", "hello", "interj")
db.put(rec, lsh)
print("exists:", db.exists(rec.label), "| count:", db.count(), "| incomplete:", db.count_incomplete())
got = db.get(rec.label); print(got)
try: got.first_encountered = "x"
except AttributeError as e: print("write to first_encountered ->", e)
try: got.set_layer(0, "x")
except PermissionError as e: print("write to layer 0 ->", e)
got.set_layer(7, [0.1, 0.2]); print("layer 7 writable, value:", got.get_layer(7))

exists: True | count: 1 | incomplete: 1
LSH_Datatype(label=41fdb0d8d378…, word='Hello')
write to first_encountered -> first_encountered is immutable (Rabies Principle).
write to layer 0 -> Layer 0 (word_surface) is acquisition-only.
layer 7 writable, value: [0.1, 0.2]


## Master label of the JSON index

In [5]:
idx = hg9.HyperIndex()
spec = hg9.ImageSpec(8, 8, hg9.PixelMode.GRAY8)
g = hg9.HyperGallery(spec); r = g.index_image(list(range(64)), "ramp"); idx.add_image_record(r)
print("master_label:", idx.master_label(), "| records:", idx.record_count())

master_label: 50afcdbcd3ff874fcc6d1828bbff95ffe08017318e0809f6455a30acc9a388c0 | records: 1
